# Clase 1 — Forward Propagation y Funciones de Activación

## Objetivos de aprendizaje
- Entender cómo una red neuronal calcula una predicción combinando pesos, sesgos y una función de activación (forward propagation).
- Implementar desde cero, con NumPy puro (sin Keras ni TensorFlow), una red neuronal genérica capa por capa.
- Ejecutar esa red con dos arquitecturas distintas y observar sus salidas.
- Comparar las funciones de activación sigmoide y ReLU, y visualizar en una gráfica el problema del *vanishing gradient*.

## Qué vamos a construir
Vamos a programar las piezas de una red neuronal —inicialización de pesos, suma ponderada, activación— y el bucle que las encadena capa por capa. Después vamos a comparar dos funciones de activación y ver, con una gráfica, por qué una de ellas facilita entrenar redes más profundas que la otra.

## Repaso teórico rápido: forward propagation

Una red neuronal es una cadena de nodos organizados en capas. Cada nodo:

1. Recibe una o varias entradas.
2. Calcula una **suma ponderada**: `z = x1*w1 + x2*w2 + ... + bias`.
3. Pasa `z` por una **función de activación** (aquí, la sigmoide `a = 1/(1+e^-z)`).
4. Entrega esa salida `a` como entrada a la siguiente capa.

Repetir esto desde la capa de entrada hasta la capa de salida es **forward propagation**. Importante: en esta clase los pesos son aleatorios — todavía no hay entrenamiento, solo el cálculo de una predicción con los pesos que la red ya tiene.

In [ ]:
import numpy as np

def initialize_network(num_inputs, num_hidden_layers, num_nodes_hidden, num_nodes_output):
    """Crea la estructura de la red: un diccionario con pesos y sesgos aleatorios por nodo."""
    num_nodes_previous = num_inputs
    network = {}

    for layer in range(num_hidden_layers + 1):
        if layer == num_hidden_layers:
            layer_name = 'output'
            num_nodes = num_nodes_output
        else:
            layer_name = 'layer_{}'.format(layer + 1)
            num_nodes = num_nodes_hidden[layer]

        network[layer_name] = {}
        for node in range(num_nodes):
            node_name = 'node_{}'.format(node + 1)
            network[layer_name][node_name] = {
                'weights': np.around(np.random.uniform(size=num_nodes_previous), decimals=2),
                'bias': np.around(np.random.uniform(size=1), decimals=2),
            }

        num_nodes_previous = num_nodes

    return network


def compute_weighted_sum(inputs, weights, bias):
    """z = producto punto entre entradas y pesos, más el sesgo."""
    return np.sum(inputs * weights) + bias


def node_activation(weighted_sum):
    """Activación sigmoide: comprime cualquier número al rango (0, 1)."""
    return 1.0 / (1.0 + np.exp(-1 * weighted_sum))


def forward_propagate(network, inputs):
    """Propaga las entradas capa por capa hasta obtener la predicción final."""
    layer_inputs = list(inputs)

    for layer in network:
        layer_data = network[layer]
        layer_outputs = []

        for layer_node in layer_data:
            node_data = layer_data[layer_node]
            node_output = node_activation(
                compute_weighted_sum(layer_inputs, node_data['weights'], node_data['bias'])
            )
            layer_outputs.append(np.around(node_output[0], decimals=4))

        if layer != 'output':
            print('Salidas de la capa oculta {}: {}'.format(layer.split('_')[1], layer_outputs))

        layer_inputs = layer_outputs

    return layer_outputs

print('Funciones definidas: initialize_network, compute_weighted_sum, node_activation, forward_propagate')

## Probemos la red con dos arquitecturas distintas

Primero, `small_network`: 5 entradas, 3 capas ocultas (3, 2 y 3 nodos) y 1 salida.

In [ ]:
np.random.seed(0)  # para que la demo sea reproducible en clase

small_network = initialize_network(
    num_inputs=5,
    num_hidden_layers=3,
    num_nodes_hidden=[3, 2, 3],
    num_nodes_output=1
)

inputs = np.around(np.random.uniform(size=5), decimals=2)
print('Entradas:', inputs)

predictions = forward_propagate(small_network, inputs)
print('Predicción final de small_network:', predictions)

In [ ]:
# Segunda red: my_network, misma cantidad de entradas pero otra arquitectura
# (2, 3 y 2 nodos ocultos, y 3 nodos de salida en vez de 1)
my_network = initialize_network(
    num_inputs=5,
    num_hidden_layers=3,
    num_nodes_hidden=[2, 3, 2],
    num_nodes_output=3
)

inputs = np.around(np.random.uniform(size=5), decimals=2)
print('Entradas:', inputs)

predictions = forward_propagate(my_network, inputs)
print('Predicción final de my_network:', predictions)

## Repaso teórico rápido: funciones de activación y vanishing gradient

- **Sigmoide** `σ(z) = 1/(1+e^-z)`: salida entre 0 y 1, derivada `σ'(z) = σ(z)(1-σ(z))`. Problema: para `z` muy grande o muy negativo, la derivada se acerca a 0 → **vanishing gradient** (el gradiente "desaparece" al propagarse hacia atrás en redes profundas).
- **ReLU** `f(x) = max(0, x)`: derivada 1 si `x>0`, 0 si `x≤0`. Mitiga el vanishing gradient porque su derivada no se aplasta en la zona positiva. Riesgo: "neuronas muertas" si la entrada es siempre negativa.

Vamos a graficar ambas funciones junto a sus derivadas para ver el problema con nuestros propios ojos.

In [ ]:
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return np.where(z > 0, 1, 0)

z = np.linspace(-10, 10, 400)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(z, sigmoid(z), label='Sigmoide')
axes[0].plot(z, sigmoid_derivative(z), label='Derivada')
axes[0].set_title('Sigmoide: la derivada se aplasta en los extremos')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(z, relu(z), label='ReLU')
axes[1].plot(z, relu_derivative(z), label='Derivada')
axes[1].set_title('ReLU: la derivada se mantiene en 1 para z > 0')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Resumen de la clase

- Forward propagation = suma ponderada + activación, encadenada capa por capa.
- Con NumPy puro puedes representar una red genérica como un diccionario de pesos y sesgos por nodo.
- La sigmoide sufre vanishing gradient en sus extremos; ReLU lo mitiga porque su derivada es constante en la zona positiva.
- Con pesos aleatorios la red ya predice algo, pero todavía no ha aprendido nada — eso es backpropagation, tema de la clase 2.

**Para seguir profundizando:**
- Capítulo del manual: [`manual/02_forward_propagation.md`](../manual/02_forward_propagation.md) y [`manual/03_funciones_activacion.md`](../manual/03_funciones_activacion.md)
- Notebook de práctica de esta clase: [`notebooks_practica/clase01_practica_forward_propagation_activaciones.ipynb`](../notebooks_practica/clase01_practica_forward_propagation_activaciones.ipynb)